Read data from bronze

In [0]:
df = spark.table("dev_project.bronze.crm_cust_info")
df.display()

Transformation

_Trimming all string valued column_

In [0]:
from pyspark.sql.functions import trim, col, StringType
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))


df.display()

Normalization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

df = df.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
    .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
    .otherwise("n/a"),
).withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
    .when(F.upper(F.col("cst_gndr")) == "F", "Female")
    .otherwise("n/a"),
)

df.display()

Renaming the column

In [0]:
name_map={
    "cst_id":"customer_id",
    "cst_key":"customer_key",
    "cst_firstname":"firstname",
    "cst_lastname":"lastname",
    "cst_marital_status":"marital_status",
    "cst_gndr":"gender",
    "cst_create_date":"create_date"
}


for old_name,new_name in name_map.items():
  df = df.withColumnRenamed(old_name,new_name)
  
df.display()

Save it to silver layer

In [0]:
df.write.mode("overwrite").saveAsTable("dev_project.silver.crm_customer")

In [0]:
%sql
select * from dev_project.silver.crm_customer